# Azure Quantum Manual Test Plan: Q#, Qiskit, and Cirq Job Submission

This is a manual test plan notebook for verifying end-to-end job submission and result correctness against Azure Quantum.

## What's Tested
- **Q# job submission** to three simulators via `target.submit()`
- **Qiskit job submission** to three simulators via `AzureQuantumProvider`
- **Cirq job submission** to three simulators via `AzureQuantumService` (including the generic Cirq→QIR submission path for targets without a dedicated Cirq target wrapper)

## Simulators Under Test

| Target | Provider |
|--------|----------|
| `ionq.simulator` | IonQ |
| `quantinuum.sim.h2-1e` | Quantinuum |
| `rigetti.sim.qvm` | Rigetti |

## Test Circuit
All tests use a 3-qubit GHZ circuit, which produces the Bell state $\frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$. Valid results will show approximately 50% `000` and 50% `111` outcomes. Each test cell ends with an assertion that validates this.

## How to Run
1. Fill in the workspace connection cell below with your Azure Quantum workspace details.
2. Run cells top to bottom within each test section.
3. Each job submission blocks until the job completes and prints a `PASS` or raises an `AssertionError`.

## Setup

Before opening this notebook, create a clean Python environment, install the local `azure-quantum` build into it, and point the notebook kernel at that environment:

1. **Create a virtual environment:**
   ```
   python -m venv {env_name}
   ```

2. **Activate it:**
   - Windows: `{env_name}\Scripts\activate`
   - macOS/Linux: `source {env_name}/bin/activate`

3. **Install the local `azure-quantum` package** (run from the root of the `azure-quantum-python` repo):
   ```
   pip install ".\azure-quantum[qsharp,qiskit,cirq]"
   ```

4. **Set the kernel for this notebook** to `{env_name}` using the kernel picker in the top-right corner of the notebook editor.

## Workspace Configuration

Set `resource_id` below to point to your Azure Quantum workspace. **This is the only cell you need to edit** — all test sections share this connection.

In [ ]:
resource_id = ""

This repo's `.env` file injects service principal credentials that are blocked by Conditional Access for interactive use. Clear them so `DefaultAzureCredential` falls through to interactive login (VS Code / Azure CLI).

In [ ]:
import os

for _var in ("AZURE_CLIENT_ID", "AZURE_CLIENT_SECRET",
             "AZURE_CLIENT_CERTIFICATE_PATH", "AZURE_CLIENT_SEND_CERTIFICATE_CHAIN"):
    os.environ.pop(_var, None)

Connect to the Azure Quantum workspace using the `resource_id` set above.

In [ ]:
from azure.quantum import Workspace

workspace = Workspace(resource_id=resource_id)

---

## Q# Job Tests

Submit a compiled Q# program to each simulator target via `target.submit()`. Results are returned as a probability dictionary, e.g. `{'[0, 0, 0]': 0.47, '[1, 1, 1]': 0.53}`.

Initialize `qsharp` with the `Base` target profile, define the GHZ operation in Q#, and compile it into a QIR payload that can be submitted directly to an Azure Quantum target.

In [ ]:
import qsharp

qsharp.init(target_profile=qsharp.TargetProfile.Base)

In [ ]:
%%qsharp

operation GHZ() : Result[] {
    use qs = Qubit[3];
    H(qs[0]);
    CNOT(qs[0], qs[1]);
    CNOT(qs[1], qs[2]);
    return MResetEachZ(qs);
}

In [ ]:
GHZ = qsharp.compile("GHZ()")

Define the validation and submission helpers for the Q# section. `validate_qsharp_ghz` asserts that both `[0, 0, 0]` and `[1, 1, 1]` outcomes are present and each accounts for roughly half of the probability mass. `submit_qsharp_job` wraps `target.submit()` for a given target name.

In [ ]:
from typing import cast
from azure.quantum.target import Target


def validate_qsharp_ghz(target_name, results, tolerance=0.35):
    """
    Validate results from a Q# GHZ job submitted via target.submit().
    Expected format: {'[0, 0, 0]': ~0.5, '[1, 1, 1]': ~0.5} (proportions, not counts).
    """
    unexpected = {k: v for k, v in results.items() if k not in ('[0, 0, 0]', '[1, 1, 1]')}
    if unexpected:
        print(f"  [{target_name}] WARN: unexpected outcomes: {unexpected}")
    prob_000 = results.get('[0, 0, 0]', 0.0)
    prob_111 = results.get('[1, 1, 1]', 0.0)
    assert prob_000 > 0, f"[{target_name}] Expected '[0, 0, 0]' in results, got: {results}"
    assert prob_111 > 0, f"[{target_name}] Expected '[1, 1, 1]' in results, got: {results}"
    assert abs(prob_000 - 0.5) < tolerance, \
        f"[{target_name}] [0,0,0] probability {prob_000:.2f} too far from 0.5 (tolerance {tolerance})"
    assert abs(prob_111 - 0.5) < tolerance, \
        f"[{target_name}] [1,1,1] probability {prob_111:.2f} too far from 0.5 (tolerance {tolerance})"
    print(f"  [{target_name}] PASS: [0,0,0]={prob_000:.1%}, [1,1,1]={prob_111:.1%}")


def submit_qsharp_job(target_name, shots=100):
    """Submit the compiled GHZ program to the given target and return the job."""
    target = cast(Target, workspace.get_targets(target_name))
    return target.submit(GHZ, name=f"ghz-qsharp-{target_name}", shots=shots)

Submit the GHZ program to all three targets in a fast serial loop (submissions are non-blocking), then wait for all responses concurrently using a thread pool. Results are validated as they arrive.

In [ ]:
import concurrent.futures

qsharp_targets = [
    "ionq.simulator",
    "quantinuum.sim.h2-1e",
    "rigetti.sim.qvm",
]

# Submit all jobs first (non-blocking)
print("Submitting Q# GHZ jobs...")
jobs = {}
for target_name in qsharp_targets:
    jobs[target_name] = submit_qsharp_job(target_name)
    print(f"  Submitted to {target_name} (id: {jobs[target_name].id})")

# Wait for all results in parallel
print("\nWaiting for results...")
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = {executor.submit(job.get_results): target_name for target_name, job in jobs.items()}
    for future in concurrent.futures.as_completed(futures):
        target_name = futures[future]
        try:
            results = future.result()
            print(f"  [{target_name}] Results: {results}")
            validate_qsharp_ghz(target_name, results)
        except Exception as exc:
            print(f"  [{target_name}] FAIL: {exc}")

---

## Qiskit Job Tests

Submit a Qiskit circuit to each simulator target via `AzureQuantumProvider`. Results come back as shot counts, e.g. `{'000': 47, '111': 53}`.

Build the equivalent 3-qubit GHZ circuit in Qiskit, which will be submitted to each target backend below.

In [ ]:
from qiskit import QuantumCircuit

# 3-qubit GHZ circuit matching the Q# GHZ operation above
circuit = QuantumCircuit(3, 3)
circuit.name = "GHZ-3"
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.measure([0, 1, 2], [0, 1, 2])
circuit.draw(output="text")

Connect to the workspace via `AzureQuantumProvider` and list the available backends to confirm the expected targets are accessible.

In [ ]:
from azure.quantum.qiskit import AzureQuantumProvider

provider = AzureQuantumProvider(workspace)
print("Available backends:")
for b in provider.backends():
    print("  -", b.name)

Define the validation and submission helpers for the Qiskit section. `validate_qiskit_ghz` asserts that both `000` and `111` shot counts are present and each accounts for roughly half of all shots. `submit_qiskit_job` wraps `backend.run()` for a given target name.

In [ ]:
def validate_qiskit_ghz(target_name, counts, tolerance=0.35):
    """
    Validate results from a Qiskit GHZ job submitted via AzureQuantumProvider.
    Expected format: {'000': ~50%, '111': ~50%} of total shots.
    """
    total = sum(counts.values())
    assert total > 0, f"[{target_name}] No results returned"
    unexpected = {k: v for k, v in counts.items() if k not in ('000', '111')}
    if unexpected:
        print(f"  [{target_name}] WARN: unexpected outcomes: {unexpected}")
    count_000 = counts.get('000', 0)
    count_111 = counts.get('111', 0)
    assert count_000 > 0, f"[{target_name}] Expected '000' in counts, got: {counts}"
    assert count_111 > 0, f"[{target_name}] Expected '111' in counts, got: {counts}"
    ratio_000 = count_000 / total
    ratio_111 = count_111 / total
    assert abs(ratio_000 - 0.5) < tolerance, \
        f"[{target_name}] '000' ratio {ratio_000:.2f} too far from 0.5 (tolerance {tolerance})"
    assert abs(ratio_111 - 0.5) < tolerance, \
        f"[{target_name}] '111' ratio {ratio_111:.2f} too far from 0.5 (tolerance {tolerance})"
    print(f"  [{target_name}] PASS: '000'={count_000} ({ratio_000:.1%}), '111'={count_111} ({ratio_111:.1%}), total={total}")


def submit_qiskit_job(target_name, shots=100):
    """Submit the Qiskit GHZ circuit to the given backend and return the job."""
    backend = provider.get_backend(target_name)
    return backend.run(circuit, shots=shots, job_name=f"ghz-qiskit-{target_name}")

Submit the GHZ circuit to all three backends in a fast serial loop (submissions are non-blocking), then wait for all responses concurrently using a thread pool. Counts are validated as they arrive.

In [ ]:
import concurrent.futures

qiskit_targets = [
    "ionq.simulator",
    "quantinuum.sim.h2-1e",
    "rigetti.sim.qvm",
]

# Submit all jobs first (non-blocking)
print("Submitting Qiskit GHZ jobs...")
jobs = {}
for target_name in qiskit_targets:
    jobs[target_name] = submit_qiskit_job(target_name)
    print(f"  Submitted to {target_name} (id: {jobs[target_name].job_id()})")

# Wait for all results in parallel
print("\nWaiting for results...")
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = {executor.submit(job.result): target_name for target_name, job in jobs.items()}
    for future in concurrent.futures.as_completed(futures):
        target_name = futures[future]
        try:
            counts = future.result().get_counts()
            print(f"  [{target_name}] Counts: {counts}")
            validate_qiskit_ghz(target_name, counts)
        except Exception as exc:
            print(f"  [{target_name}] FAIL: {exc}")

---

## Cirq Job Tests

Submit a Cirq circuit to each simulator target via `AzureQuantumService`. Results come back as a `cirq.Result` with per-shot measurement arrays, e.g. `{'m': [[0,0,0], [1,1,1], ...]}`.

Build the 3-qubit GHZ circuit in Cirq with measurement key `m`, which will be submitted to each target below. Also define `validate_cirq_ghz`, which asserts that both `000` and `111` outcomes are present and each accounts for roughly half of all shots.

In [ ]:
import cirq
from collections import Counter

cirq_repetitions = 200
q0, q1, q2 = cirq.LineQubit.range(3)

cirq_circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.CNOT(q1, q2),
    cirq.measure(q0, q1, q2, key="m"),
)

print(cirq_circuit)


def validate_cirq_ghz(target_name, result, repetitions, tolerance=0.35):
    """
    Validate results from a Cirq GHZ job submitted via AzureQuantumService.
    Expects measurement key 'm' with ~50% '000' and ~50% '111' shots.
    """
    meas = result.measurements.get("m")
    assert meas is not None, (
        f"[{target_name}] Missing measurement key 'm'. "
        f"Available keys: {sorted(result.measurements.keys())}"
    )
    counts = Counter("".join(str(int(b)) for b in row) for row in meas)
    total = sum(counts.values())
    assert total == repetitions, (
        f"[{target_name}] Expected {repetitions} shots, got {total}. Counts: {counts}"
    )
    unexpected = {k: v for k, v in counts.items() if k not in ("000", "111")}
    if unexpected:
        print(f"  [{target_name}] WARN: unexpected outcomes: {unexpected}")
    count_000 = counts.get("000", 0)
    count_111 = counts.get("111", 0)
    assert count_000 > 0, f"[{target_name}] Expected '000' in results, got: {counts}"
    assert count_111 > 0, f"[{target_name}] Expected '111' in results, got: {counts}"
    ratio_000 = count_000 / total
    ratio_111 = count_111 / total
    assert abs(ratio_000 - 0.5) < tolerance, (
        f"[{target_name}] '000' ratio {ratio_000:.2f} too far from 0.5 (tolerance {tolerance})"
    )
    assert abs(ratio_111 - 0.5) < tolerance, (
        f"[{target_name}] '111' ratio {ratio_111:.2f} too far from 0.5 (tolerance {tolerance})"
    )
    print(
        f"  [{target_name}] PASS: '000'={count_000} ({ratio_000:.1%}), "
        f"'111'={count_111} ({ratio_111:.1%}), total={total}"
    )

Connect to the workspace via `AzureQuantumService` and list the available Cirq target wrappers to confirm the expected targets are accessible.

In [ ]:
from azure.quantum.cirq import AzureQuantumService

cirq_service = AzureQuantumService(workspace)

print("Cirq target wrappers available in workspace:")
for t in cirq_service.targets():
    print(f"  - {t.name}: {type(t).__name__}")

Submit the GHZ circuit to all three targets in a fast serial loop (submissions are non-blocking), then wait for all responses concurrently using a thread pool. Results are validated as they arrive.

In [ ]:
import concurrent.futures

cirq_targets = [
    "ionq.simulator",
    "quantinuum.sim.h2-1e",
    "rigetti.sim.qvm",
]

# Submit all jobs first (non-blocking)
print("Submitting Cirq GHZ jobs...")
cirq_jobs = {}
for target_name in cirq_targets:
    cirq_jobs[target_name] = cirq_service.create_job(
        program=cirq_circuit,
        repetitions=cirq_repetitions,
        target=target_name,
        name=f"ghz-cirq-{target_name}",
    )
    print(f"  Submitted to {target_name} (id: {cirq_jobs[target_name].job_id()})")

# Wait for all results in parallel
print("\nWaiting for results...")
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = {executor.submit(job.results): target_name for target_name, job in cirq_jobs.items()}
    for future in concurrent.futures.as_completed(futures):
        target_name = futures[future]
        try:
            result = future.result()
            # cirq_ionq>=1.6 returns a list of results even for a single circuit.
            if isinstance(result, (list, tuple)):
                if len(result) != 1:
                    raise ValueError(f"[{target_name}] Expected a single result, got {len(result)}")
                result = result[0]
            # The IonQ provider wrapper (cirq_ionq.Job) returns a SimulatorResult or
            # QPUResult rather than a cirq.Result. Normalize by calling to_cirq_result().
            if hasattr(result, "to_cirq_result"):
                result = result.to_cirq_result()
            validate_cirq_ghz(target_name, result, repetitions=cirq_repetitions)
        except Exception as exc:
            print(f"  [{target_name}] FAIL: {exc}")